In [1]:
import sys
import os

# Aggiunge la cartella radice del progetto al percorso di ricerca di Python
sys.path.append(os.path.abspath(os.path.join('..')))

In [4]:
import sentence_transformers
print(sentence_transformers.__version__)

model = SentenceTransformer()
print(model)           # se è vuoto, il caricamento è fallito
print(list(model.modules()))  # deve avere almeno Transformer + Pooling

3.2.1
SentenceTransformer(
  (0): None
)
[SentenceTransformer(
  (0): None
)]


In [ ]:
# Cella 1 - pulisci i file .pyc corrotti e reinstalla
import subprocess
subprocess.run(["find", "/opt/conda", "-name", "*.pyc", "-delete"])
subprocess.run(["pip", "install", "--force-reinstall", "--no-cache-dir", 
                "torch", "sentence-transformers"])

In [5]:
import app.core.config as config
#import app.core.json_prompt_read as json_prompt_read
from sentence_transformers import SentenceTransformer


class PromptEmbedder:
    def __init__(self, model):
        self.model = model

    def embed(self, prompt):
        # Use the model to generate an embedding for the prompt
        embedding = self.model.encode(prompt)
        return embedding

if __name__ == "__main__":

    # Load the model specified in the config
    model_name = config.EMBEDDING_MODEL
    model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

    embedder = PromptEmbedder(model)
    #json_path = "prompt.json"

    #prompt = json_prompt_read.read_json_prompt(json_path)
    prompt = "ricetta con uova e 300kcal"

    # Get the embedding for the prompt
    query_embedding = embedder.embed(prompt)
  


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 24627.37it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
###CONFRONTA CON I DATI NEL DB
import chromadb
from chromadb.config import Settings

client = chromadb.HttpClient(
    host="chroma",        # nome del servizio nel compose
    port=8000
)

collection = client.get_collection("ricette")
print(collection.count())

Exception: {"error":"TypeError(\"object of type 'int' has no len()\")"}

In [6]:
# L'embedding del prompt è già calcolato nella cella precedente
results = collection.query(
    query_embeddings=[query_embedding.tolist()],  # converti numpy array in lista
    n_results=10,                           # top 10 più simili
    include=["documents", "distances", "metadatas"]
)

# Stampa i risultati
for i, (doc, distance) in enumerate(zip(results["documents"][0], results["distances"][0])):
    similarity = 1 - distance  # ChromaDB restituisce la distanza, non la similarità
    print(f"\n--- Risultato {i+1} (similarità: {similarity:.4f}) ---")
    print(doc)

Exception: {"error":"TypeError(\"object of type 'int' has no len()\")"}

In [ ]:
client = chromadb.HttpClient(host="localhost", port=8000)